In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob

import itertools
from pybedtools import BedTool

from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
from scipy.stats import binomtest, ttest_ind

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'

donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Skin', 'Inflamed', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
s = 'Start'
tmp_clusts = [f'{s}_{x}' for x in donors]

In [ ]:
### use 500bp bins
def process_bed(s):
    bins_mc = pd.read_csv(f'{projdir}/bed/bins/rna_cluster_donor/{s}.chrom500_CGN-Both.sparse.bed.gz', sep='\t', header=None)
    bins_mc.index = [f'{x}_{y}' for x,y in zip(bins_mc[0], bins_mc[1])]
    bins_mc['sample'] = s
    return bins_mc

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(process_bed, tmp_clusts), total=len(tmp_clusts)))

In [ ]:
mcg = [x[4] / x[5] for x in results]
tmp_samples = pd.concat([x['sample'] for x in results])

In [ ]:
merged = pd.DataFrame(pd.concat(mcg))
merged.columns = ['mCG']
merged['sample'] = tmp_samples
merged['cluster'] = [x.split('_')[0] for x in merged['sample']]
merged['donor'] = [x.split('_')[1] for x in merged['sample']]
merged.shape

In [ ]:
frac = 0.1
tmp_merged = merged.sample(frac=frac)
tmp_merged.shape

In [ ]:
tmp_merged.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(6, 5))

x_lab = 'mCG'
hue = 'donor'

ax = sns.kdeplot(data=tmp_merged, x=x_lab, hue=hue)
                 
ax.set_xlabel('CpG Methylation', fontsize=12)
ax.set_ylabel('', fontsize=12)
ax.set_title('', fontsize=20)
ax.grid(False)

ax.tick_params(axis='x', labelrotation=45)

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2'
tmp_merged.to_csv(f'{outdir}/start_donor_bins_mcg_dist.csv.gz', sep='\t')

In [ ]:
!date